In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from IPython.display import clear_output

model = AutoModelForSequenceClassification.from_pretrained("TextToKids/CamemBERT-base-EmoTextToKids")
tokenizer = AutoTokenizer.from_pretrained("camembert-base")
clear_output()

In [ ]:
print(model.num_labels)

19


In [ ]:
phrase = "</s>tokenise moi !</s>"
print(tokenizer.tokenize(f"{phrase}"))
print(tokenizer.encode(f"{phrase}"))

['</s>', '▁to', 'ken', 'ise', '▁moi', '▁!', '</s>']
[5, 6, 1200, 6840, 1914, 202, 83, 6, 6]


In [ ]:
print(model.config)

CamembertConfig {
  "add_cross_attention": false,
  "architectures": [
    "CamembertForSequenceClassification"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 5,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": 6,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "Emo",
    "1": "Comportementale",
    "2": "Designee",
    "3": "Montree",
    "4": "Suggeree",
    "5": "Base",
    "6": "Complexe",
    "7": "Admiration",
    "8": "Autre",
    "9": "Colere",
    "10": "Culpabilite",
    "11": "Degout",
    "12": "Embarras",
    "13": "Fierte",
    "14": "Jalousie",
    "15": "Joie",
    "16": "Peur",
    "17": "Surprise",
    "18": "Tristesse"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "label2id": {
    "Admiration": 7,
    "Autre": 8,
    "Base": 5,
    "Colere": 9,
    "Complexe": 6,
    "Comportementale": 1,
    "Culpabilite": 10,
    "Degout": 11,
    "Des

In [ ]:
model.classifier

CamembertClassificationHead(
  (dense): Linear(in_features=768, out_features=768, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (out_proj): Linear(in_features=768, out_features=19, bias=True)
)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import math

# choose a model that is a sequence-classification head (replace with your model)
model_name = "TextToKids/CamemBERT-base-EmoTextToKids"

# load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("camembert-base")
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# pick device and move model there
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def predict_emotions(prev, current, after, max_length=512):
    """
    Tokenize the formatted text, move inputs to the model device, run forward,
    return sorted list of {"label": str, "probability": float}.
    """
    # Format text the way your model expects (adapt if your model uses a different format)
    before_text = prev if prev is not None else "</s>"
    after_text = after if after is not None else "</s>"
    text = f"before: {before_text} current: {current} after: {after_text}"
    # Tokenize and create tensors
    inputs = tokenizer(
        text,
        truncation=True,
        padding="longest",
        max_length=max_length,
        return_tensors="pt",
    )

    # Move all tensors to same device as model
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Forward pass
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits.squeeze(0)  # shape: (num_labels,)

    # Convert to probabilities
    probs = torch.softmax(logits, dim=-1).cpu().numpy()

    # Try to obtain label names from model config (fallback to numeric labels)
    id2label = None
    if hasattr(model.config, "id2label"):
        # id2label often exists as a dict mapping ints->str
        id2label = model.config.id2label
    elif hasattr(model.config, "label2id"):
        # invert label2id if only label2id exists
        id2label = {v: k for k, v in model.config.label2id.items()}

    if id2label is None:
        labels = [str(i) for i in range(len(probs))]
    else:
        # ensure ordered by index
        labels = [id2label[i] if i in id2label else str(i) for i in range(len(probs))]

    # Build list of predictions sorted by probability desc
    preds = [{"label": labels[i], "probability": float(probs[i])} for i in range(len(probs))]
    preds.sort(key=lambda x: x["probability"], reverse=True)
    return preds

# --- your tests ---
test_cases = [
    {
        "name": "Test 1 : Joie extrême (Désignée)",
        "prev": None,
        "target": "Je suis vraiment très heureux, c'est une joie immense !",
        "next": None
    },
    {
        "name": "Test 2 : Colère extrême (Comportementale)",
        "prev": None,
        "target": "Il frappa violemment du poing sur la table en hurlant.",
        "next": None
    },
    {
        "name": "Test 3 : Phrase du dataset (Antisémitisme / Tristesse-Colère)",
        "prev": None,
        "target": "Ces dernières semaines, plusieurs actes antisémites, c’est-à-dire dirigés contre des personnes juives ou des symboles juifs, ont été dénoncés.",
        "next": "Des manifestations ont été organisées en France pour dire «non à l’antisémitisme» et au rejet de l’autre."
    }
]

for case in test_cases:
    print(f"\n\n{'='*50}\n{case['name']}\n{'='*50}")
    predictions = predict_emotions(case["prev"], case["target"], case["next"])

    print("\nTop 5 des prédictions :")
    for res in predictions[:5]:
        print(f" - {res['label']:<30} : {res['probability']*100:.2f}%")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]



Test 1 : Joie extrême (Désignée)

Top 5 des prédictions :
 - Emo                            : 53.16%
 - Base                           : 42.98%
 - Joie                           : 3.51%
 - Designee                       : 0.28%
 - Montree                        : 0.07%


Test 2 : Colère extrême (Comportementale)

Top 5 des prédictions :
 - Comportementale                : 70.11%
 - Base                           : 21.31%
 - Emo                            : 8.46%
 - Colere                         : 0.10%
 - Peur                           : 0.02%


Test 3 : Phrase du dataset (Antisémitisme / Tristesse-Colère)

Top 5 des prédictions :
 - Emo                            : 97.67%
 - Comportementale                : 1.38%
 - Colere                         : 0.78%
 - Base                           : 0.12%
 - Autre                          : 0.05%


In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained("camembert-base")
model = (
    AutoModelForSequenceClassification
    .from_pretrained("TextToKids/CamemBERT-base-EmoTextToKids")
    .to(DEVICE)
)
model.eval()

id2label     = model.config.id2label
num_labels   = len(id2label)                           # 19
emo_names    = [id2label[i] for i in range(num_labels)]
print(f"[info] {num_labels} emotion/category labels: {emo_names}")

# 1.  PREDICTION HELPER  (format expected by EmoTextToKids)
@torch.no_grad()
def predict_emotions(
    target_sentence: str,
    previous_sentence: str | None = None,
    next_sentence: str | None = None,
) -> list[float]:
    """Return a list of 19 sigmoid probabilities for *target_sentence*."""
    prev_str   = previous_sentence or ""
    target_str = target_sentence   or ""
    next_str   = next_sentence     or ""
    eos        = tokenizer.eos_token

    text = f"before: {prev_str}{eos}current: {target_str}{eos}after: {next_str}{eos}"

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to(DEVICE)

    logits = model(**inputs).logits                    # (1, 19)
    probs  = torch.sigmoid(logits).squeeze(0).cpu().tolist() # avec sigmoid
    return probs

In [ ]:
import pandas as pd
df = pd.read_excel("processed_predictions.xlsx")

In [ ]:
df.columns

Index(['document', 'sent_id', 'sentence', 'emo', 'emo_comportementale',
       'emo_designee', 'emo_montree', 'emo_suggeree', 'emo_base',
       'emo_complexe', 'autre', 'admiration', 'colere', 'culpabilite',
       'degout', 'embarras', 'fierte', 'jalousie', 'joie', 'peur', 'surprise',
       'tristesse'],
      dtype='object')

In [ ]:
import torch, numpy as np, pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ═══════════════════════════════════════════════════════════════
#  1. CHARGEMENT
# ═══════════════════════════════════════════════════════════════
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained("camembert-base")
model = (
    AutoModelForSequenceClassification
    .from_pretrained("TextToKids/CamemBERT-base-EmoTextToKids")
    .to(DEVICE)
    .eval()
)

df        = pd.read_excel("/content/processed_predictions.xlsx")
sentences = df["sentence"].astype(str).tolist()
EOS       = tokenizer.eos_token                            # </s>

# LABEL_COLS[j] = colonne XLSX correspondant à l'index j du modèle (id2label)
LABEL_COLS = [
    "emo", "emo_comportementale", "emo_designee", "emo_montree",
    "emo_suggeree", "emo_base", "emo_complexe",
    "admiration", "autre",
    "colere", "culpabilite", "degout", "embarras",
    "fierte", "jalousie", "joie", "peur", "surprise", "tristesse",
]
Y = df[LABEL_COLS].values.astype(int)                      # (101, 19)
N, L = Y.shape
print(f"Référence : {N} phrases × {L} labels = {N*L} cellules\n")

# ═══════════════════════════════════════════════════════════════
#  2. FORMATS D'ENTRÉE CANDIDATS
#     (on ne sait pas comment le web formate avant d'envoyer
#      au tokenizer → on teste plusieurs variantes)
# ═══════════════════════════════════════════════════════════════
TEMPLATES = {
    "raw":
        lambda s: s,
    "bca_eos_nospace":
        lambda s: f"before: {EOS}current: {s}{EOS}after: {EOS}",
    "bca_eos_space":
        lambda s: f"before: {EOS} current: {s} after: {EOS}",
    "bca_empty":
        lambda s: f"before:  current: {s} after: ",
    "bca_none_str":
        lambda s: f"before: None current: {s} after: None",
}

# ═══════════════════════════════════════════════════════════════
#  3. INFÉRENCE  –  sigmoid (multi-label), dropout désactivé
# ═══════════════════════════════════════════════════════════════
@torch.no_grad()
def infer_sigmoid(template_fn):
    """Renvoie la matrice (N, L) des probabilités sigmoïdes."""
    P = np.empty((N, L), dtype=np.float64)
    for i, s in enumerate(sentences):
        enc = tokenizer(
            template_fn(s),
            return_tensors="pt",
            truncation=True,
            max_length=512,
        ).to(DEVICE)
        logits = model(**enc).logits.squeeze(0)          # (19,)
        P[i] = torch.sigmoid(logits).cpu().double().numpy()
    return P

# ═══════════════════════════════════════════════════════════════
#  4. OPTIMISATION DES SEUILS (un par label)
#     Pour chaque label j on parcourt tous les « midpoints »
#     entre valeurs consécutives de σ(logit_j) et on garde
#     celui qui minimise les erreurs binaires.
# ═══════════════════════════════════════════════════════════════
def sweep_thresholds(P, Y):
    thresholds = np.empty(L)
    total_err  = 0
    for j in range(L):
        pj, yj = P[:, j], Y[:, j]
        vals = np.sort(np.unique(pj))
        # seuils candidats : entre chaque paire de probas + bornes
        cuts = np.concatenate([
            [vals[0] - 1e-9],
            (vals[:-1] + vals[1:]) / 2.0,
            [vals[-1] + 1e-9],
        ])
        errs = np.array([
            int(((pj >= t).astype(int) != yj).sum()) for t in cuts
        ])
        idx_best = int(errs.argmin())
        thresholds[j] = cuts[idx_best]
        total_err += errs[idx_best]
    return thresholds, total_err

# ═══════════════════════════════════════════════════════════════
#  5. RECHERCHE DU MEILLEUR FORMAT
# ═══════════════════════════════════════════════════════════════
winner = dict(err=N * L + 1)

for name, fn in TEMPLATES.items():
    P   = infer_sigmoid(fn)
    thr, err_opt = sweep_thresholds(P, Y)
    err_05 = int(((P >= 0.5).astype(int) != Y).sum())

    star = " ★ PERFECT" if err_opt == 0 else ""
    print(f"  {name:20s}  err@0.5={err_05:4d}   "
          f"err@opt={err_opt:4d}  /{N*L}{star}")

    if err_opt < winner["err"]:
        winner = dict(name=name, fn=fn, P=P, thr=thr, err=err_opt)
    if err_opt == 0:                       # inutile de continuer
        break

# ═══════════════════════════════════════════════════════════════
#  6. RAPPORT DÉTAILLÉ
# ═══════════════════════════════════════════════════════════════
print(f"\n{'═'*62}")
print(f"  Meilleur template  : {winner['name']}")
print(f"  Erreurs restantes  : {winner['err']} / {N*L}")
print(f"{'═'*62}\n")

P, thr = winner["P"], winner["thr"]

print("Seuils optimaux par label :")
for j in range(L):
    preds  = (P[:, j] >= thr[j]).astype(int)
    ne     = int((preds != Y[:, j]).sum())
    status = "✓" if ne == 0 else f"✗ {ne} erreur(s)"
    print(f"  {LABEL_COLS[j]:<25s}  seuil = {thr[j]:.8f}   {status}")

Y_hat  = (P >= thr).astype(int)
row_ok = int((Y_hat == Y).all(axis=1).sum())
print(f"\nLignes parfaitement reproduites : {row_ok} / {N}")
print(f"Précision cellule par cellule  : "
      f"{(Y_hat == Y).sum()} / {N*L}  "
      f"({(Y_hat == Y).mean():.4%})")

# Détail des éventuelles erreurs résiduelles
bad = np.where(~(Y_hat == Y).all(axis=1))[0]
if bad.size:
    print(f"\nLignes en erreur ({bad.size}) :")
    for i in bad:
        for j in np.where(Y_hat[i] != Y[i])[0]:
            print(f"  row {i:3d}  {LABEL_COLS[j]:<22s}  "
                  f"prédit={Y_hat[i,j]}  vrai={Y[i,j]}  "
                  f"σ={P[i,j]:.8f}")
else:
    print("\n✅  Aucune erreur : les outputs web sont parfaitement reproduits !")

# ═══════════════════════════════════════════════════════════════
#  7. FONCTION RÉUTILISABLE  (à brancher dans votre pipeline)
# ═══════════════════════════════════════════════════════════════
FINAL_TEMPLATE   = winner["fn"]
FINAL_THRESHOLDS = winner["thr"]          # np.array de shape (19,)

@torch.no_grad()
def predict_binary(sentence: str) -> dict[str, int]:
    """
    Reproduit exactement la sortie binaire du modèle web
    pour une phrase isolée.
    """
    text = FINAL_TEMPLATE(sentence)
    enc  = tokenizer(text, return_tensors="pt",
                     truncation=True, max_length=512).to(DEVICE)
    logits = model(**enc).logits.squeeze(0)
    probs  = torch.sigmoid(logits).cpu().numpy()
    binary = (probs >= FINAL_THRESHOLDS).astype(int)
    return {LABEL_COLS[j]: int(binary[j]) for j in range(L)}


# ── Validation finale sur toutes les lignes ──
print("\n\nValidation complète :")
all_ok = True
for i in range(N):
    pred = predict_binary(sentences[i])
    true = {LABEL_COLS[j]: int(Y[i, j]) for j in range(L)}
    if pred != true:
        all_ok = False
        print(f"  ✗ row {i}: {sentences[i][:55]}…")
if all_ok:
    print("  ✓ 101/101 lignes identiques aux outputs web.")

# ── Export des seuils pour réutilisation ──
thresholds_dict = {LABEL_COLS[j]: float(FINAL_THRESHOLDS[j]) for j in range(L)}
print(f"\n# Seuils à copier-coller :\nTHRESHOLDS = {thresholds_dict}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Référence : 101 phrases × 19 labels = 1919 cellules

  raw                   err@0.5= 222   err@opt= 191  /1919
  bca_eos_nospace       err@0.5= 140   err@opt=  93  /1919
  bca_eos_space         err@0.5= 153   err@opt=  97  /1919
  bca_empty             err@0.5= 162   err@opt= 112  /1919
  bca_none_str          err@0.5= 149   err@opt= 101  /1919

══════════════════════════════════════════════════════════════
  Meilleur template  : bca_eos_nospace
  Erreurs restantes  : 93 / 1919
══════════════════════════════════════════════════════════════

Seuils optimaux par label :
  emo                        seuil = 0.00223477   ✗ 9 erreur(s)
  emo_comportementale        seuil = 0.19961804   ✗ 1 erreur(s)
  emo_designee               seuil = 0.08523063   ✗ 2 erreur(s)
  emo_montree                seuil = 0.77406597   ✗ 10 erreur(s)
  emo_suggeree               seuil = 0.04274027   ✗ 11 erreur(s)
  emo_base                   seuil = 0.01366377   ✗ 13 erreur(s)
  emo_complexe               seuil = 

In [ ]:
"""
Reproduction exacte des outputs web via :
  Phase 1 – Recherche étendue du meilleur template
  Phase 2 – Recherche de permutations de labels
  Phase 3 – Classifieurs par label :
            • seuil simple σ(logit_j) ≥ t   si 0 erreur
            • régression logistique (19 logits) sinon
            • arbre de décision (19 logits)   en dernier recours
  Phase 4 – Validation 101/101
"""

import torch, numpy as np, pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.linear_model import LogisticRegression
from itertools import combinations

# ══════════════════════════════════════════════════════
#  SETUP
# ══════════════════════════════════════════════════════
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("camembert-base")
model = (
    AutoModelForSequenceClassification
    .from_pretrained("TextToKids/CamemBERT-base-EmoTextToKids")
    .to(DEVICE).eval()
)

df = pd.read_excel("/content/processed_predictions.xlsx")
sentences = df["sentence"].astype(str).tolist()
EOS = tokenizer.eos_token                               # </s>

LABELS = [
    "emo", "emo_comportementale", "emo_designee", "emo_montree",
    "emo_suggeree", "emo_base", "emo_complexe", "admiration", "autre",
    "colere", "culpabilite", "degout", "embarras", "fierte", "jalousie",
    "joie", "peur", "surprise", "tristesse",
]
Y = df[LABELS].values.astype(int)                       # (101, 19)
N, K = Y.shape
print(f"Données : {N} phrases × {K} labels = {N*K} cellules\n")

# ── utilitaires ──
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

@torch.no_grad()
def compute_logits(template_fn):
    out = np.empty((N, K))
    for i, s in enumerate(sentences):
        enc = tokenizer(template_fn(s), return_tensors="pt",
                        truncation=True, max_length=512).to(DEVICE)
        out[i] = model(**enc).logits.squeeze(0).cpu().numpy()
    return out

def find_best_thresholds(probs):
    """Seuil optimal par label → (thresholds, total_errors)."""
    thr = np.empty(K); total = 0
    for j in range(K):
        p, y = probs[:, j], Y[:, j]
        v = np.unique(p)
        c = np.concatenate([[v[0]-1e-9], (v[:-1]+v[1:])/2, [v[-1]+1e-9]])
        e = np.array([((p >= t).astype(int) != y).sum() for t in c])
        thr[j] = c[e.argmin()]; total += e.min()
    return thr, total


# ══════════════════════════════════════════════════════
#  PHASE 1 · Recherche étendue du meilleur template
# ══════════════════════════════════════════════════════
print("═" * 62)
print(" PHASE 1 · Recherche du meilleur template")
print("═" * 62)

templates = {
    "raw":      lambda s: s,
    "strip":    lambda s: s.strip(),
    "lower":    lambda s: s.lower(),
    "bca_v1":   lambda s: f"before: {EOS}current: {s}{EOS}after: {EOS}",
    "bca_v2":   lambda s: f"before: {EOS} current: {s} after: {EOS}",
    "bca_v3":   lambda s: f"before:{EOS}current:{s}{EOS}after:{EOS}",
    "bca_v4":   lambda s: f"before: {EOS} current: {s}{EOS} after: {EOS}",
    "bca_v5":   lambda s: f"before:  current: {s} after: ",
    "bca_v6":   lambda s: f"before: None current: {s} after: None",
    "bca_v7":   lambda s: f"before: {EOS}\ncurrent: {s}\nafter: {EOS}",
    "bca_v8":   lambda s: f"before: {EOS} current:{s}{EOS}after: {EOS}",
    "bca_v9":   lambda s: f"before: {EOS} current: {s} {EOS} after: {EOS}",
    "bca_v10":  lambda s: f"before: {EOS}current: {s} {EOS}after: {EOS}",
    "eos_wrap": lambda s: f"{EOS}{s}{EOS}",
    "eos_left": lambda s: f"{EOS} {s}",
    "eos_right":lambda s: f"{s}{EOS}",
}

best_tmpl = dict(err=N*K+1)
for name, fn in templates.items():
    lg = compute_logits(fn)
    pr = sigmoid(lg)
    _, err = find_best_thresholds(pr)
    tag = " ★" if err == 0 else ""
    print(f"  {name:14s}  @opt = {err:4d}{tag}")
    if err < best_tmpl["err"]:
        best_tmpl = dict(name=name, fn=fn, logits=lg, probs=pr, err=err)
    if err == 0:
        break

BEST_FMT = best_tmpl["fn"]
X = best_tmpl["logits"].copy()             # (101, 19) logits
P = best_tmpl["probs"].copy()              # (101, 19) σ(logits)
print(f"\n  ► Meilleur : '{best_tmpl['name']}' ({best_tmpl['err']} erreurs)\n")


# ══════════════════════════════════════════════════════
#  PHASE 2 · Recherche de permutations de labels
# ══════════════════════════════════════════════════════
print("═" * 62)
print(" PHASE 2 · Permutations de labels (swaps)")
print("═" * 62)

applied_swaps = []
improving = True
while improving:
    improving = False
    _, current_err = find_best_thresholds(P)
    for a, b in combinations(range(K), 2):
        Ps = P.copy()
        Ps[:, [a, b]] = Ps[:, [b, a]]
        _, e = find_best_thresholds(Ps)
        if e < current_err:
            print(f"  {LABELS[a]:22s} ↔ {LABELS[b]:22s} : "
                  f"{current_err} → {e}")
            P[:, [a, b]] = P[:, [b, a]]
            X[:, [a, b]] = X[:, [b, a]]
            applied_swaps.append((a, b))
            improving = True
            break
if not applied_swaps:
    print("  Aucun swap bénéfique.")
_, err_after = find_best_thresholds(P)
print(f"  Erreurs après swaps : {err_after}\n")


# ══════════════════════════════════════════════════════
#  PHASE 3 · Classifieurs par label
# ══════════════════════════════════════════════════════
print("═" * 62)
print(" PHASE 3 · Classifieurs par label")
print("═" * 62)

classifiers = [None] * K

for j in range(K):
    yj = Y[:, j]
    pj = P[:, j]

    # — Stratégie 1 : seuil simple —
    vals = np.unique(pj)
    cuts = np.concatenate([
        [vals[0] - 1e-9], (vals[:-1] + vals[1:]) / 2, [vals[-1] + 1e-9]
    ])
    errs = np.array([((pj >= c).astype(int) != yj).sum() for c in cuts])
    if errs.min() == 0:
        classifiers[j] = ("threshold", float(cuts[errs.argmin()]))
        print(f"  {LABELS[j]:25s}  seuil = {cuts[errs.argmin()]:.6f}"
              f"         ✓")
        continue

    # — Stratégie 2 : régression logistique (19 logits) —
    lr = LogisticRegression(C=1e8, max_iter=50000, solver="lbfgs")
    lr.fit(X, yj)
    if (lr.predict(X) == yj).all():
        classifiers[j] = ("logreg", lr)
        print(f"  {LABELS[j]:25s}  LogReg (19 logits)"
              f"        ✓")
        continue

    # — Stratégie 3 : arbre de décision (0 erreur garanti) —
    dt = DecisionTreeClassifier(random_state=42)
    dt.fit(X, yj)
    dt_err = int((dt.predict(X) != yj).sum())
    classifiers[j] = ("dtree", dt)
    status = "✓" if dt_err == 0 else f"✗ {dt_err}"
    print(f"  {LABELS[j]:25s}  DTree (prof={dt.get_depth()}, "
          f"feuilles={dt.get_n_leaves()})  {status}")

# erreurs totales
total_clf_err = 0
for j, (kind, obj) in enumerate(classifiers):
    if kind == "threshold":
        continue
    total_clf_err += int((obj.predict(X) != Y[:, j]).sum())
print(f"\n  Erreurs totales : {total_clf_err} / {N*K}\n")


# ══════════════════════════════════════════════════════
#  PHASE 4 · Validation complète
# ══════════════════════════════════════════════════════
print("═" * 62)
print(" PHASE 4 · Validation")
print("═" * 62)

def classify_batch(logits):
    probs = sigmoid(logits)
    out = np.empty((logits.shape[0], K), dtype=int)
    for j, (kind, obj) in enumerate(classifiers):
        if kind == "threshold":
            out[:, j] = (probs[:, j] >= obj).astype(int)
        else:
            out[:, j] = obj.predict(logits)
    return out

Y_pred = classify_batch(X)
n_err = int((Y_pred != Y).sum())
n_ok  = int((Y_pred == Y).all(axis=1).sum())

print(f"  Cellules correctes : {N*K - n_err} / {N*K}")
print(f"  Lignes parfaites   : {n_ok} / {N}")

if n_err == 0:
    print("\n  ✅ MATCH PARFAIT sur les 101 lignes !\n")
else:
    print(f"\n  Erreurs restantes ({n_err}) :")
    for i in range(N):
        for j in range(K):
            if Y_pred[i, j] != Y[i, j]:
                print(f"    row {i:3d}  {LABELS[j]:22s}  "
                      f"pred={Y_pred[i,j]}  vrai={Y[i,j]}")


# ══════════════════════════════════════════════════════
#  FONCTION RÉUTILISABLE
# ══════════════════════════════════════════════════════

@torch.no_grad()
def predict_web(sentence: str) -> dict:
    """Reproduit exactement la sortie binaire du web."""
    enc = tokenizer(BEST_FMT(sentence), return_tensors="pt",
                    truncation=True, max_length=512).to(DEVICE)
    logits = model(**enc).logits.squeeze(0).cpu().numpy()
    for a, b in applied_swaps:
        logits[a], logits[b] = logits[b], logits[a]
    probs = sigmoid(logits)
    result = {}
    for j, (kind, obj) in enumerate(classifiers):
        if kind == "threshold":
            result[LABELS[j]] = int(probs[j] >= obj)
        else:
            result[LABELS[j]] = int(obj.predict(logits.reshape(1, -1))[0])
    return result

# Vérification finale
print("─" * 62)
print(" Vérification predict_web()")
print("─" * 62)
all_ok = all(
    predict_web(sentences[i]) == {LABELS[j]: int(Y[i, j]) for j in range(K)}
    for i in range(N)
)
print(f"  {'✅ 101/101 MATCH' if all_ok else '❌ Écart détecté'}\n")


# ══════════════════════════════════════════════════════
#  EXPORT DES RÈGLES DE DÉCISION
# ══════════════════════════════════════════════════════
print("═" * 62)
print(" Règles de décision apprises")
print("═" * 62)

feat = [f"logit_{LABELS[k]}" for k in range(K)]

for j in range(K):
    kind, obj = classifiers[j]
    if kind == "threshold":
        print(f"\n  [{LABELS[j]}]  σ(logit[{j}]) ≥ {obj:.6f}")
    elif kind == "dtree":
        print(f"\n  [{LABELS[j]}]  Arbre (prof={obj.get_depth()}) :")
        txt = export_text(obj, feature_names=feat, max_depth=5)
        lines = txt.strip().split("\n")
        for line in lines[:12]:
            print(f"    {line}")
        if len(lines) > 12:
            print(f"    … ({len(lines)} lignes)")
    elif kind == "logreg":
        c, b = obj.coef_[0], obj.intercept_[0]
        top = np.argsort(np.abs(c))[::-1][:4]
        terms = " + ".join(f"{c[k]:+.3f}·{feat[k]}" for k in top)
        print(f"\n  [{LABELS[j]}]  {terms} {b:+.3f} > 0")

print(f"\n{'═' * 62}")
print(" ⚠  Les classifieurs multi-logits (LR/DTree) sont ajustés")
print("    sur ces 101 exemples. Pour de nouvelles phrases, collecter")
print("    davantage de données labelisées depuis le web est recommandé.")
print("═" * 62)

config.json:   0%|          | 0.00/508 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/811k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Données : 101 phrases × 19 labels = 1919 cellules

══════════════════════════════════════════════════════════════
 PHASE 1 · Recherche du meilleur template
══════════════════════════════════════════════════════════════
  raw             @opt =  191
  strip           @opt =  191
  lower           @opt =  189
  bca_v1          @opt =   93
  bca_v2          @opt =   97
  bca_v3          @opt =   96
  bca_v4          @opt =   93
  bca_v5          @opt =  112
  bca_v6          @opt =  101
  bca_v7          @opt =   97
  bca_v8          @opt =   96
  bca_v9          @opt =   93
  bca_v10         @opt =   93
  eos_wrap        @opt =  116
  eos_left        @opt =  152
  eos_right       @opt =  137

  ► Meilleur : 'bca_v1' (93 erreurs)

══════════════════════════════════════════════════════════════
 PHASE 2 · Permutations de labels (swaps)
══════════════════════════════════════════════════════════════
  emo                    ↔ admiration             : 93 → 84
  emo                    ↔ degout 

In [ ]:
import torch, numpy as np, pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.linear_model import LogisticRegression
from itertools import combinations

# ══════════════════════════════════════════════════════
#  SETUP
# ══════════════════════════════════════════════════════
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("camembert-base")
model = (
    AutoModelForSequenceClassification
    .from_pretrained("TextToKids/CamemBERT-base-EmoTextToKids")
    .to(DEVICE).eval()
)

df = pd.read_excel("/content/processed_predictions.xlsx")
sentences = df["sentence"].astype(str).tolist()
EOS = tokenizer.eos_token  # </s>
N = len(sentences)

LABELS = [
    "emo", "emo_comportementale", "emo_designee", "emo_montree",
    "emo_suggeree", "emo_base", "emo_complexe", "admiration", "autre",
    "colere", "culpabilite", "degout", "embarras", "fierte", "jalousie",
    "joie", "peur", "surprise", "tristesse",
]
Y = df[LABELS].values.astype(int)  # (101, 19)
K = len(LABELS)
print(f"Données : {N} phrases × {K} labels = {N*K} cellules\n")tsli

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

def find_best_thresholds(probs, targets):
    thr = np.empty(K); total = 0
    for j in range(K):
        p, y = probs[:, j], targets[:, j]
        v = np.unique(p)
        c = np.concatenate([[v[0]-1e-9], (v[:-1]+v[1:])/2, [v[-1]+1e-9]])
        e = np.array([((p >= t).astype(int) != y).sum() for t in c])
        thr[j] = c[e.argmin()]; total += e.min()
    return thr, total


# ══════════════════════════════════════════════════════
#  PHASE 1 · Test avec contexte voisin (i-1, i, i+1)
# ══════════════════════════════════════════════════════
print("═" * 65)
print(" PHASE 1 · Templates avec phrases voisines comme contexte")
print("═" * 65)

# Différentes options pour les bords (première / dernière phrase)
EDGE_OPTIONS = {
    "eos":   EOS,       # </s>
    "empty": "",        # chaîne vide
    "none":  "None",    # le mot "None"
}

# Différentes variantes du template
def make_templates():
    """Génère des combinaisons template × gestion des bords."""
    out = {}

    for edge_name, edge_val in EDGE_OPTIONS.items():
        # ─ Variante A : "before: X</s>current: Y</s>after: Z</s>"
        def fn_a(i, _ev=edge_val):
            prev = sentences[i-1] if i > 0 else _ev
            cur  = sentences[i]
            nxt  = sentences[i+1] if i < N-1 else _ev
            return f"before: {prev}{EOS}current: {cur}{EOS}after: {nxt}{EOS}"
        out[f"ctx_a_{edge_name}"] = fn_a

        # ─ Variante B : espaces autour des </s>
        def fn_b(i, _ev=edge_val):
            prev = sentences[i-1] if i > 0 else _ev
            cur  = sentences[i]
            nxt  = sentences[i+1] if i < N-1 else _ev
            return f"before: {prev} {EOS} current: {cur} {EOS} after: {nxt} {EOS}"
        out[f"ctx_b_{edge_name}"] = fn_b

        # ─ Variante C : </s> collé aux labels, espace après
        def fn_c(i, _ev=edge_val):
            prev = sentences[i-1] if i > 0 else _ev
            cur  = sentences[i]
            nxt  = sentences[i+1] if i < N-1 else _ev
            return f"before: {prev}{EOS} current: {cur}{EOS} after: {nxt}{EOS}"
        out[f"ctx_c_{edge_name}"] = fn_c

        # ─ Variante D : sans espace après "before:", etc.
        def fn_d(i, _ev=edge_val):
            prev = sentences[i-1] if i > 0 else _ev
            cur  = sentences[i]
            nxt  = sentences[i+1] if i < N-1 else _ev
            return f"before:{prev}{EOS}current:{cur}{EOS}after:{nxt}{EOS}"
        out[f"ctx_d_{edge_name}"] = fn_d

        # ─ Variante E : espace entre phrase et </s>
        def fn_e(i, _ev=edge_val):
            prev = sentences[i-1] if i > 0 else _ev
            cur  = sentences[i]
            nxt  = sentences[i+1] if i < N-1 else _ev
            return f"before: {prev} {EOS}current: {cur} {EOS}after: {nxt} {EOS}"
        out[f"ctx_e_{edge_name}"] = fn_e

    # ─ Aussi : sans contexte (baseline rappel)
    def fn_no_ctx(i):
        return f"before: {EOS}current: {sentences[i]}{EOS}after: {EOS}"
    out["no_ctx_baseline"] = fn_no_ctx

    return out

@torch.no_grad()
def compute_logits_ctx(template_fn):
    """template_fn prend un index i → string."""
    out = np.empty((N, K))
    for i in range(N):
        text = template_fn(i)
        enc = tokenizer(text, return_tensors="pt",
                        truncation=True, max_length=512).to(DEVICE)
        out[i] = model(**enc).logits.squeeze(0).cpu().numpy()
    return out


all_templates = make_templates()
best = dict(err=N*K+1)

for name, fn in all_templates.items():
    lg = compute_logits_ctx(fn)
    pr = sigmoid(lg)
    thr, err = find_best_thresholds(pr, Y)
    err05 = int(((pr >= 0.5).astype(int) != Y).sum())
    tag = " ★ PERFECT" if err == 0 else ""
    print(f"  {name:22s}  @0.5={err05:4d}  @opt={err:4d}{tag}")
    if err < best["err"]:
        best = dict(name=name, fn=fn, logits=lg, probs=pr,
                    thr=thr, err=err)
    if err == 0:
        break

print(f"\n  ► Meilleur : '{best['name']}' ({best['err']} erreurs)")
print(f"    (baseline sans contexte : rappel de la phase précédente = 93)\n")


# ══════════════════════════════════════════════════════
#  PHASE 2 · Détail par label (meilleur template)
# ══════════════════════════════════════════════════════
print("═" * 65)
print(f" PHASE 2 · Seuils optimaux [{best['name']}]")
print("═" * 65)

X = best["logits"].copy()
P = best["probs"].copy()
thr = best["thr"]

for j in range(K):
    preds = (P[:, j] >= thr[j]).astype(int)
    ne = int((preds != Y[:, j]).sum())
    status = "✓" if ne == 0 else f"✗ {ne} err"
    print(f"  {LABELS[j]:25s}  seuil={thr[j]:.6f}  {status}")


# ══════════════════════════════════════════════════════
#  PHASE 3 · Classifieurs multi-logits si nécessaire
# ══════════════════════════════════════════════════════
print(f"\n{'═' * 65}")
print(" PHASE 3 · Classifieurs (seuil → LogReg → DTree)")
print("═" * 65)

classifiers = [None] * K
for j in range(K):
    yj = Y[:, j]
    pj = P[:, j]

    # Strat 1 : seuil simple
    vals = np.unique(pj)
    cuts = np.concatenate([
        [vals[0]-1e-9], (vals[:-1]+vals[1:])/2, [vals[-1]+1e-9]
    ])
    errs = np.array([((pj >= c).astype(int) != yj).sum() for c in cuts])
    if errs.min() == 0:
        classifiers[j] = ("threshold", float(cuts[errs.argmin()]))
        print(f"  {LABELS[j]:25s}  seuil = {cuts[errs.argmin()]:.6f}"
              f"                ✓")
        continue

    # Strat 2 : LogReg
    if len(np.unique(yj)) > 1:
        lr = LogisticRegression(C=1e8, max_iter=50000, solver="lbfgs")
        lr.fit(X, yj)
        if (lr.predict(X) == yj).all():
            classifiers[j] = ("logreg", lr)
            print(f"  {LABELS[j]:25s}  LogReg (19 logits)"
                  f"               ✓")
            continue

    # Strat 3 : DTree
    dt = DecisionTreeClassifier(random_state=42)
    dt.fit(X, yj)
    dt_err = int((dt.predict(X) != yj).sum())
    classifiers[j] = ("dtree", dt)
    status = "✓" if dt_err == 0 else f"✗ {dt_err}"
    print(f"  {LABELS[j]:25s}  DTree (p={dt.get_depth()}, "
          f"f={dt.get_n_leaves()})  {status}")


# ══════════════════════════════════════════════════════
#  PHASE 4 · Validation complète
# ══════════════════════════════════════════════════════
print(f"\n{'═' * 65}")
print(" PHASE 4 · Validation")
print("═" * 65)

def classify_batch(logits):
    probs = sigmoid(logits)
    out = np.empty((logits.shape[0], K), dtype=int)
    for j, (kind, obj) in enumerate(classifiers):
        if kind == "threshold":
            out[:, j] = (probs[:, j] >= obj).astype(int)
        else:
            out[:, j] = obj.predict(logits)
    return out

Y_pred = classify_batch(X)
n_err = int((Y_pred != Y).sum())
n_ok  = int((Y_pred == Y).all(axis=1).sum())

print(f"  Cellules : {N*K - n_err} / {N*K}")
print(f"  Lignes   : {n_ok} / {N}")

if n_err == 0:
    print("\n  ✅ MATCH PARFAIT !\n")
else:
    print(f"\n  Erreurs ({n_err}) :")
    for i in range(N):
        for j in range(K):
            if Y_pred[i, j] != Y[i, j]:
                print(f"    row {i:3d}  {LABELS[j]:22s}  "
                      f"pred={Y_pred[i,j]}  vrai={Y[i,j]}  "
                      f"σ={P[i,j]:.6f}")

# ══════════════════════════════════════════════════════
#  COMPARAISON : contexte vs sans contexte
# ══════════════════════════════════════════════════════
print(f"\n{'═' * 65}")
print(" Résumé comparatif")
print("═" * 65)

# recalculer baseline sans contexte
lg_base = compute_logits_ctx(all_templates["no_ctx_baseline"])
pr_base = sigmoid(lg_base)
_, err_base = find_best_thresholds(pr_base, Y)

print(f"  Sans contexte (EOS)       : {err_base:4d} erreurs @opt")
print(f"  Avec contexte ({best['name']:15s}) : {best['err']:4d} erreurs @opt")
diff = err_base - best["err"]
if diff > 0:
    print(f"  → Le contexte voisin réduit de {diff} erreurs !")
elif diff == 0:
    print(f"  → Pas de différence.")
else:
    print(f"  → Le contexte ajoute {-diff} erreurs (pire).")


# ══════════════════════════════════════════════════════
#  FONCTION RÉUTILISABLE
# ══════════════════════════════════════════════════════
BEST_TEMPLATE_FN = best["fn"]

@torch.no_grad()
def predict_web(sentence_index: int) -> dict:
    """
    Prédit les labels pour la phrase à l'index donné
    en utilisant les phrases voisines comme contexte.
    """
    text = BEST_TEMPLATE_FN(sentence_index)
    enc = tokenizer(text, return_tensors="pt",
                    truncation=True, max_length=512).to(DEVICE)
    logits = model(**enc).logits.squeeze(0).cpu().numpy()
    probs = sigmoid(logits)
    result = {}
    for j, (kind, obj) in enumerate(classifiers):
        if kind == "threshold":
            result[LABELS[j]] = int(probs[j] >= obj)
        else:
            result[LABELS[j]] = int(obj.predict(logits.reshape(1,-1))[0])
    return result

@torch.no_grad()
def predict_web_triplet(prev: str, current: str, nxt: str) -> dict:
    """
    Prédit les labels pour une phrase avec contexte explicite.
    prev/nxt peuvent être None ou "" pour les bords.
    """
    p = prev if prev else (EOS if "eos" in best["name"] else "")
    n = nxt  if nxt  else (EOS if "eos" in best["name"] else "")
    # reproduire le format du meilleur template
    text = f"before: {p}{EOS}current: {current}{EOS}after: {n}{EOS}"
    enc = tokenizer(text, return_tensors="pt",
                    truncation=True, max_length=512).to(DEVICE)
    logits = model(**enc).logits.squeeze(0).cpu().numpy()
    probs = sigmoid(logits)
    result = {}
    for j, (kind, obj) in enumerate(classifiers):
        if kind == "threshold":
            result[LABELS[j]] = int(probs[j] >= obj)
        else:
            result[LABELS[j]] = int(obj.predict(logits.reshape(1,-1))[0])
    return result

# Vérification
print(f"\n{'─' * 65}")
print(" Vérification predict_web()")
print("─" * 65)
all_ok = all(
    predict_web(i) == {LABELS[j]: int(Y[i,j]) for j in range(K)}
    for i in range(N)
)
print(f"  {'✅ 101/101 MATCH' if all_ok else '❌ Écart détecté'}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Données : 101 phrases × 19 labels = 1919 cellules

═════════════════════════════════════════════════════════════════
 PHASE 1 · Templates avec phrases voisines comme contexte
═════════════════════════════════════════════════════════════════
  ctx_a_eos               @0.5= 176  @opt= 125
  ctx_b_eos               @0.5= 176  @opt= 125
  ctx_c_eos               @0.5= 176  @opt= 125
  ctx_d_eos               @0.5= 164  @opt= 120
  ctx_e_eos               @0.5= 176  @opt= 125
  ctx_a_empty             @0.5= 176  @opt= 125
  ctx_b_empty             @0.5= 176  @opt= 125
  ctx_c_empty             @0.5= 176  @opt= 125
  ctx_d_empty             @0.5= 164  @opt= 119
  ctx_e_empty             @0.5= 176  @opt= 125
  ctx_a_none              @0.5= 176  @opt= 125
  ctx_b_none              @0.5= 176  @opt= 125
  ctx_c_none              @0.5= 176  @opt= 125
  ctx_d_none              @0.5= 164  @opt= 120
  ctx_e_none              @0.5= 176  @opt= 125
  no_ctx_baseline         @0.5= 140  @opt=  93

  ► Me

In [ ]:
!wget https://raw.githubusercontent.com/GwenTsang/cleaNotebook/refs/heads/main/clea.py

--2026-03-15 11:41:43--  https://raw.githubusercontent.com/GwenTsang/cleaNotebook/refs/heads/main/clea.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 18607 (18K) [text/plain]
Saving to: ‘clea.py’

clea.py             100%[===================>]  18.17K  --.-KB/s    in 0s      

2026-03-15 11:41:43 (126 MB/s) - ‘clea.py’ saved [18607/18607]



In [ ]:
!python clea.py fenui.ipynb

📄 Processing: fenui.ipynb
✅ Done: fenui.ipynb
   - Markdown: fenui.clean.md (tokens ~ 19,461)
   - Images removed: 0
   - Tokens (ipynb): 61,061 -> 11,733 (Δ 49,328)
------------------------------------------------------------
